# Costas label — firing-rate check & selection-bias diagnostic (permutation)

Three questions to settle before comparing **category-coding strength** between Costas `exc` and
`inh`. Reads only the saved CSVs (no NWB loading). **All tests are non-parametric permutation tests.**

**Q1 — Do the two Costas classes have the same firing rate?**  The classifier is pure waveform and
never sees rate, so recovering the textbook "interneurons fire faster" difference would be
out-of-sample evidence the split tracks a real cell class. Finding no difference is equally
informative: it means either the labels are weak in MTL, or human MTL does not show the pattern.

**Q2 — Did the category-cell selection prefer high-firing cells?**  Detection power scales with spike
count. Measured within each class separately.

**Q3 — Was that preference STRONGER in one class?**  The decision. Selection is class-*blind*, but
that does not make it class-*neutral*: if it filtered one class harder on spike count, the two
selected subsets sit in different parts of their effect-size distributions and a naive strength
comparison is confounded.

## The stats

No distributional assumptions, no mixed models. Each test shuffles labels thousands of times and asks
how often chance alone produces something as extreme as the observed value.

**Every shuffle is WITHIN SUBJECT.** Subjects differ in both their firing-rate distributions and their
exc/inh mix, so shuffling across subjects would manufacture differences that are really
between-patient variation. Restricted (within-subject) permutation is what replaces the mixed model.

| | statistic | what gets shuffled | null |
|---|---|---|---|
| **Q1**  | subject-stratified `AUC = P(inh > exc)` | `celltype` within subject | the classes have the same firing rate |
| **Q1b** | median over subjects of `median(inh) − median(exc)` | **sign flip** per subject | no consistent direction across subjects |
| **Q2**  | rank-biserial `r` of selected vs not, within class | `selective` within subject × celltype | selection ignores spike count |
| **Q3**  | `Δ = r_inh − r_exc` | `selective` within subject × **spike-count decile** | given spike count and subject, selection does not depend on class |

Q3 must stratify on spike count. Shuffling `celltype` instead would test "class is unrelated to
selection", which a Q1 rate difference alone would reject — telling you nothing about asymmetry.

**Reading strength:** the p-value is not the effect size. With ~856 units a trivial difference gives
a tiny p. Read the **AUC** (0.5 = nothing, 0.56 small, 0.64 medium, 0.71 large, 0.75+ visibly
separated) and the **count of subjects going the same direction**.

000673, hippocampus + amygdala, Costas `costas_class` only.

## Section 0 — Setup

In [ ]:
# [0.1] imports, config
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata

PROJECT = Path.cwd()                        # run from E:\SBCAT\celltyping
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

RS = 42
N_PERM = 10000
plt.rcParams['figure.dpi'] = 100

# ---- CONFIG ----
DATASET      = '000673'
CT           = PROJECT / 'outputs' / 'celltype'
SELECT_AREAS = ['hippocampus', 'amygdala']
region_tag   = '+'.join(a.lower() for a in SELECT_AREAS)
CAT_CELLS_CSV = CT / f'category_cells_{region_tag}.csv'          # the paper-method selection
COSTAS_CSV    = CT / f'unit_labels_costas_{region_tag}.csv'      # costas_class (exc/inh)
MASTER_CSV    = CT / f'master_units_{DATASET}.csv'               # mean_rate_hz
STIM_WIN      = (0.2, 1.0)                  # window the selection test counted spikes in
GROUPS        = ['inh', 'exc']
GRP_COL       = {'inh': '#d62728', 'exc': '#1f77b4'}
ALPHA         = 0.05
N_DECILES     = 10                          # spike-count strata for the Q3 null
MIN_WIN_SPIKES = 0                          # QC: set e.g. 50 to drop near-silent units (0 = keep all)
print(f'{DATASET} | {SELECT_AREAS} | costas_class {GROUPS} | {N_PERM} perms | seed {RS}')

In [ ]:
# [0.2] merge selection + Costas label + rate metrics  (one row per tested, labelled unit)
sel_csv = pd.read_csv(CAT_CELLS_CSV)
cos = pd.read_csv(COSTAS_CSV)[['unit_id', 'costas_class', 'costas_class_conf']]
mas = pd.read_csv(MASTER_CSV)[['unit_ID', 'mean_rate_hz']].rename(columns={'unit_ID': 'unit_id'})
d = (sel_csv.merge(cos, on='unit_id', how='left')
            .merge(mas, on='unit_id', how='left')
            .rename(columns={'costas_class': 'celltype'}))
d = d[d.tested & d.celltype.notna()].copy()

# spikes the permutation test actually had to work with = the honest power proxy
d['win_spikes'] = d.resp_rate * (STIM_WIN[1] - STIM_WIN[0]) * d.n_pres

# Stimulus window = 0.2-1.0 s after picture onset (Encoding1/2/3 + Probe). The picture stays on
# screen 2.0 s, so this window sits entirely INSIDE the presentation - the stimulus is on for all of
# it. Not the task's Response epoch (timestamps_Response, the button press).
#
# Only TWO metrics, not three: resp_rate (mean Hz in that window) is win_spikes / (0.8 s * n_pres),
# and n_pres is constant within a session, so the two are the same quantity up to a per-session scale
# factor and give identical rank-based statistics. Keep the spike COUNT - it is what sets detection
# power and it reads directly. Session-wide rate is genuinely different (it includes inter-trial time).
RATE_METRICS = [('mean_rate_hz', 'session-wide rate (Hz)'),      # Q1 reads this (intrinsic firing)
                ('win_spikes',   'spikes in stimulus window')]   # Q2/Q3 read this (detection power)

n0 = len(d)
if MIN_WIN_SPIKES > 0:
    d = d[d.win_spikes >= MIN_WIN_SPIKES].copy()
d = d.reset_index(drop=True)

print(f'{len(d)} tested units with a Costas label' + (f'  ({n0-len(d)} dropped by QC)' if n0 != len(d) else ''))
print('  by class       :', d.celltype.value_counts().to_dict())
print('  category cells :', d[d.selective].celltype.value_counts().to_dict())
print('  base rate      : ' + '  '.join(f'{g} {100*(d.celltype==g).mean():.1f}%' for g in GROUPS))
print('  selected mix   : ' + '  '.join(f'{g} {100*(d[d.selective].celltype==g).mean():.1f}%' for g in GROUPS))
print(f'  subjects       : {d.subject.nunique()} '
      f'({d.groupby(["subject","celltype"]).size().unstack().dropna().shape[0]} with both classes)')
for col, lab in RATE_METRICS:
    nn = int(d[col].isna().sum())
    if nn:
        print(f'  [note] {nn} unit(s) missing {lab} — excluded from that row only')
print(f'  [note] {int((d.win_spikes < 50).sum())} unit(s) fire <50 spikes in-window in total '
      f'(cannot test as selective); MIN_WIN_SPIKES={MIN_WIN_SPIKES}')

In [ ]:
# [0.3] permutation machinery — stratified AUC + within-stratum label shuffling
def _prep(x, strata):
    # Per-stratum (indices, midranks). Ranks are invariant to shuffling labels WITHIN a stratum,
    # so compute them once and reuse across every permutation.
    x = np.asarray(x, float); strata = np.asarray(strata)
    ok = np.isfinite(x)
    out = []
    for s in pd.unique(strata[ok]):
        k = np.where(ok & (strata == s))[0]
        if k.size >= 2:
            out.append((k, rankdata(x[k])))
    return out

def _strat_auc(prep, m):
    # Pair-count-weighted mean of per-stratum AUC = P(group A > group B). 0.5 = no difference.
    m = np.asarray(m, bool); num = den = 0.0
    for k, r in prep:
        mk = m[k]; na = int(mk.sum()); nb = k.size - na
        if na == 0 or nb == 0:
            continue
        auc = (r[mk].sum() - na * (na + 1) / 2.0) / (na * nb)
        num += auc * na * nb; den += na * nb
    return num / den if den > 0 else np.nan

def _shuffle_within(m, strata, rng):
    # Permute the label vector inside each stratum, leaving stratum sizes and composition intact.
    m = np.asarray(m, bool).copy(); strata = np.asarray(strata)
    for s in pd.unique(strata):
        k = np.where(strata == s)[0]
        if k.size > 1:
            m[k] = rng.permutation(m[k])
    return m

def _perm_p(obs, null, center):
    # Two-sided; the +1 keeps p strictly positive (chance can never beat an observation 0% of the time).
    null = np.asarray(null, float); null = null[np.isfinite(null)]
    if not np.isfinite(obs) or null.size == 0:
        return np.nan
    return float((np.sum(np.abs(null - center) >= abs(obs - center)) + 1) / (null.size + 1))

def perm_auc(x, m, strata, shuffle_strata=None, n_perm=N_PERM, seed=RS):
    # Observed subject-stratified AUC + its within-stratum permutation null.
    # shuffle_strata defaults to `strata`; Q3 shuffles on a finer grid than it aggregates over.
    x = np.asarray(x, float); m = np.asarray(m, bool)
    strata = np.asarray(strata); ss = strata if shuffle_strata is None else np.asarray(shuffle_strata)
    prep = _prep(x, strata)
    obs = _strat_auc(prep, m)
    rng = np.random.default_rng(seed)
    null = np.array([_strat_auc(prep, _shuffle_within(m, ss, rng)) for _ in range(n_perm)])
    return obs, _perm_p(obs, null, 0.5), null
print('permutation helpers ready')

## Section 1 — Q1: do the two Costas classes have the same firing rate?

**Null:** they do. **Statistic:** `AUC = P(random inh cell fires faster than random exc cell)`,
computed within each subject and pair-count weighted. **Null distribution:** shuffle the exc/inh
label within each subject, 10,000 times.

The left panel *is* the test — if the red line sits inside the grey null, we do not reject. The right
panel is the consistency check: one bar per subject, and how many fall on each side of zero.

In [ ]:
# [1.1] Q1 — one statistic, one null, one verdict
ok = d.mean_rate_hz.notna()
auc, p, null = perm_auc(d.loc[ok, 'mean_rate_hz'], d.loc[ok, 'celltype'] == 'inh', d.loc[ok, 'subject'])
inh_med = d.loc[ok & (d.celltype == 'inh'), 'mean_rate_hz'].median()
exc_med = d.loc[ok & (d.celltype == 'exc'), 'mean_rate_hz'].median()

# subject level: one number per subject, sign-flipped for its null (nesting cannot inflate this)
pair = (d[ok].groupby(['subject', 'celltype']).mean_rate_hz.median()
             .unstack().reindex(columns=GROUPS).dropna())
diffs = (pair['inh'] - pair['exc']).to_numpy(float)
n_sub = diffs.size; n_up = int((diffs > 0).sum())
rng = np.random.default_rng(RS)
obs_med = float(np.median(diffs))
null_sf = np.array([np.median(diffs * rng.choice([-1.0, 1.0], n_sub)) for _ in range(N_PERM)])
p_sign = _perm_p(obs_med, null_sf, 0.0)
reject = bool(np.isfinite(p) and p < ALPHA)

print(f'  inh {inh_med:.2f} Hz    vs    exc {exc_med:.2f} Hz     '
      f'(n = {int((ok & (d.celltype=="inh")).sum())} / {int((ok & (d.celltype=="exc")).sum())})')
print(f'  AUC = {auc:.3f}   (0.5 = identical)          permutation p = {p:.4g}')
print(f'  subjects: {n_up}/{n_sub} have inh faster     sign-flip p = {p_sign:.4g}')
print()
print('  ' + ('REJECT the null — the classes DIFFER in firing rate.' if reject else
              'DO NOT REJECT — no evidence the Costas classes differ in firing rate.'))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.hist(null, bins=60, color='#c9c9c9', edgecolor='none')
lo, hi = np.percentile(null[np.isfinite(null)], [2.5, 97.5])
a1.axvspan(lo, hi, color='k', alpha=0.08, label='null 95%')
a1.axvline(0.5, color='k', lw=0.8, ls=':')
a1.axvline(auc, color='#d62728', lw=2.5, label=f'observed AUC = {auc:.3f}')
a1.set_xlabel('AUC  =  P(inh fires faster than exc)'); a1.set_ylabel('permutations')
a1.set_title(f'null: exc/inh label shuffled within subject   (p = {p:.3g})', fontsize=10)
a1.legend(fontsize=8)

order = np.argsort(diffs)
a2.bar(np.arange(n_sub), diffs[order],
       color=[GRP_COL['inh'] if v > 0 else GRP_COL['exc'] for v in diffs[order]], alpha=0.85)
a2.axhline(0, color='k', lw=1)
a2.set_xlabel(f'subject (sorted)    —    {n_up}/{n_sub} above zero = inh faster')
a2.set_ylabel('median(inh) − median(exc)   (Hz)')
a2.set_title(f'per-subject consistency   (sign-flip p = {p_sign:.3g})', fontsize=10)

fig.suptitle('Q1 — do the Costas classes differ in firing rate?', fontsize=12)
fig.tight_layout(); plt.show()

## Section 2 — Q2/Q3: was selection spike-count dependent, and asymmetric?

`r = 2·AUC − 1` (0 = selection ignored spike count, +1 = every selected cell above every unselected
one), aggregated over subjects. **Q2** shuffles `selective` within subject × class. **Q3** shuffles
`selective` within subject × **spike-count decile**, pooling both classes — so spike count and
subject are held fixed and only a residual class effect can move `Δ`.

In [ ]:
# [2.1] Q2 — selection shift within each class (subject-stratified, within-subject permutation)
rows = []
for g in GROUPS:
    sub = d[d.celltype == g]
    for col, lab in RATE_METRICS:
        ok_ = sub[col].notna()
        a_, p_, _ = perm_auc(sub.loc[ok_, col], sub.loc[ok_, 'selective'], sub.loc[ok_, 'subject'])
        rows.append(dict(celltype=g, metric=lab,
                         n_sel=int(sub.loc[ok_, 'selective'].sum()),
                         n_non=int((~sub.loc[ok_, 'selective']).sum()),
                         sel_med=sub.loc[ok_ & sub.selective, col].median(),
                         non_med=sub.loc[ok_ & ~sub.selective, col].median(),
                         rank_biserial=2 * a_ - 1 if np.isfinite(a_) else np.nan, perm_p=p_))
shift = pd.DataFrame(rows)
print('Q2 — did selection prefer high-spike-count cells, within each class?')
print(shift.round(4).to_string(index=False))
print('\n(rank_biserial 0 = selection ignored spike count; >0 = selected cells fired more)')

In [ ]:
# [2.2] Q3 — is the shift ASYMMETRIC?  null: selection depends on spike count + subject, not on class
PRIMARY_SEL = 'spikes in stimulus window'
SEL_COL = dict((l, c) for c, l in RATE_METRICS)[PRIMARY_SEL]

def _q3(col):
    okc = d[col].notna().to_numpy()
    x = d[col].to_numpy(float); sel = d.selective.to_numpy(bool)
    ct = d.celltype.to_numpy(); subj = d.subject.to_numpy()
    # aggregate per class over SUBJECT strata; ranks are fixed under the shuffle, so prep once
    prep = {g: (_prep(x[okc & (ct == g)], subj[okc & (ct == g)]), np.where(okc & (ct == g))[0])
            for g in GROUPS}

    def delta(s):
        r = {g: 2 * _strat_auc(p_, s[idx]) - 1 for g, (p_, idx) in prep.items()}
        return r['inh'] - r['exc'], r

    # shuffle strata = subject x spike-count decile (class deliberately NOT in the strata)
    dec = pd.qcut(pd.Series(x).rank(method='first'), N_DECILES, labels=False, duplicates='drop')
    ss = (pd.Series(subj).astype(str) + '|' + pd.Series(dec).astype(str)).to_numpy()

    obs, r_obs = delta(sel)
    rng_ = np.random.default_rng(RS)
    nulls = np.array([delta(_shuffle_within(sel, ss, rng_))[0] for _ in range(N_PERM)])
    lo_, hi_ = np.percentile(nulls[np.isfinite(nulls)], [2.5, 97.5])
    return dict(metric=dict(RATE_METRICS)[col], r_inh=r_obs['inh'], r_exc=r_obs['exc'],
                delta=obs, null_lo95=lo_, null_hi95=hi_, perm_p=_perm_p(obs, nulls, 0.0))

q3_tab = pd.DataFrame([_q3(c) for c, _ in RATE_METRICS])
print('Q3 — delta = r_inh - r_exc, vs a null where selection depends on spike count + subject, not class:')
print(q3_tab.round(4).to_string(index=False))
print('\n(delta inside the null 95% band = selection treated both classes alike)')

In [ ]:
# [2.3] selected vs non-selected spike counts, within each class (the Q2/Q3 picture)
fig, axes = plt.subplots(1, len(GROUPS), figsize=(5.2 * len(GROUPS), 3.6), sharex=True, sharey=True)
for ax, g in zip(np.atleast_1d(axes), GROUPS):
    sub = d[d.celltype == g]
    for key, mask, col_, ls in (('category cell', sub.selective, GRP_COL[g], '-'),
                                ('not selected', ~sub.selective, '#999', '--')):
        v = np.log10(sub.loc[mask, SEL_COL].clip(lower=1)).dropna()
        ax.hist(v, bins=25, alpha=0.5, color=col_, density=True, label=f'{key} (n={v.size})')
        ax.axvline(v.median(), color=col_, ls=ls, lw=1.8)
    r = shift[(shift.celltype == g) & (shift.metric == PRIMARY_SEL)].iloc[0]
    ax.set_title(f'{g}:  r = {r.rank_biserial:+.3f}   (perm p = {r.perm_p:.2g})', fontsize=10)
    ax.set_xlabel(f'log10 {PRIMARY_SEL}'); ax.legend(fontsize=7)
np.atleast_1d(axes)[0].set_ylabel('density')
fig.suptitle('Q2/Q3 — did selection pull in high-count cells, and equally for both classes?', fontsize=11)
fig.tight_layout(); plt.show()

## Section 3 — Verdict + save

In [ ]:
# [3.1] the three answers, the branch decision, and the saved diagnostic
q2 = {g: shift[(shift.celltype == g) & (shift.metric == PRIMARY_SEL)].iloc[0] for g in GROUPS}
q3 = q3_tab[q3_tab.metric == PRIMARY_SEL].iloc[0]
symmetric = bool(q3.null_lo95 <= q3.delta <= q3.null_hi95)

print('=' * 78)
print(f'COSTAS RATE DIAGNOSTIC — {DATASET} {region_tag}   ({len(d)} units, '
      f'{int(d.selective.sum())} category cells, {N_PERM} permutations)')
print('=' * 78)
print(f'Q1   same firing rate? : inh {inh_med:.2f} Hz vs exc {exc_med:.2f} Hz   '
      f'AUC={auc:.3f}   perm p={p:.4g}   -> ' + ('REJECT' if reject else 'DO NOT REJECT'))
print(f'Q1b  subject direction : {n_up}/{n_sub} subjects inh faster   sign-flip p={p_sign:.4g}')
for g in GROUPS:
    print(f'Q2   selection [{g:3s}]    : {q2[g].sel_med:.0f} vs {q2[g].non_med:.0f} spikes   '
          f'r={q2[g].rank_biserial:+.3f}   perm p={q2[g].perm_p:.4g}')
print(f'Q3   asymmetry         : delta={q3.delta:+.3f}   null 95% '
      f'[{q3.null_lo95:+.3f}, {q3.null_hi95:+.3f}]   perm p={q3.perm_p:.4g}')
print('-' * 78)
if symmetric:
    print('BRANCH -> SYMMETRIC. Selection treated both classes alike.')
    print('  Compare strength directly on the category cells: per-cell best_auc,')
    print('  subject-stratified AUC(inh vs exc), celltype shuffled within subject for the null.')
else:
    print('BRANCH -> ASYMMETRIC. Selection filtered one class harder on spike count.')
    print('  Keep the category cells and the framing, but select on ODD presentations and')
    print('  measure strength on EVEN, so the strength estimate is unbiased by construction.')
print('=' * 78)

out = CT / f'costas_rate_diagnostic_{region_tag}.csv'
q1_tab = pd.DataFrame([dict(metric='session-wide rate (Hz)', inh_med=inh_med, exc_med=exc_med,
                            auc_inh_gt_exc=auc, perm_p=p, n_subjects=n_sub,
                            n_subjects_inh_faster=n_up, sign_flip_p=p_sign, reject=reject)])
diag = pd.concat([q1_tab.assign(block='Q1_same_firing_rate'),
                  shift.assign(block='Q2_selection_shift'),
                  q3_tab.assign(block='Q3_asymmetry')], ignore_index=True)
diag.to_csv(out, index=False)
prov = dict(dataset=DATASET, areas=SELECT_AREAS, label='costas_class', stim_win=list(STIM_WIN),
            stats='within-subject restricted permutation; subject-stratified AUC / rank-biserial',
            n_perm=N_PERM, seed=RS, alpha=ALPHA, n_deciles=N_DECILES,
            min_win_spikes=MIN_WIN_SPIKES, metric_q1='session-wide rate (Hz)',
            metric_selection=PRIMARY_SEL,
            n_units=int(len(d)), n_category_cells=int(d.selective.sum()),
            q1_auc=float(auc), q1_p=float(p), q1_reject=reject,
            q1b_subjects=f'{n_up}/{n_sub}', q1b_p=float(p_sign),
            q2_r={g: float(q2[g].rank_biserial) for g in GROUPS},
            q2_p={g: float(q2[g].perm_p) for g in GROUPS},
            q3_delta=float(q3.delta), q3_null95=[float(q3.null_lo95), float(q3.null_hi95)],
            q3_p=float(q3.perm_p), symmetric=symmetric)
out.with_suffix('.json').write_text(json.dumps(prov, indent=2, default=float))
print(f'wrote {out.name} + .json')
diag

## Reading the result

**Which rate metric answers which question.**

- `mean_rate_hz` — **session-wide rate**: `n_spikes / (last spike − first spike)` over the whole
  recording, inter-trial time included (`features.py`).
- `win_spikes = resp_rate × 0.8 s × n_pres` — **total spikes in the stimulus window** (0.2–1.0 s after
  picture onset, Encoding1/2/3 + Probe), i.e. the spike count the permutation test actually had.

There is deliberately **no third metric**. `resp_rate` (mean Hz in that window) is `win_spikes`
divided by `0.8 s × n_pres`, and `n_pres` is constant within a session — so it is the same quantity
up to a per-session scale factor and gives identical rank-based statistics. Only the spike count is
kept: it is what sets detection power, and it reads directly.

Measured from the trials table, a picture stays on screen **2.016 s** (5th–95th pct 2.005–2.017), so
the 0.2–1.0 s window lies **entirely within the presentation** — the stimulus is on for all of it, in
100% of trials. Hence *stimulus window*. It is not the task's **Response** epoch
(`timestamps_Response`, the button press) either.

**Q1 reads session-wide rate** — "interneurons fire faster" is a claim about intrinsic firing.
**Q2/Q3 read `win_spikes`** — detection power is set by spikes inside the tested window, not by how
much a cell fired between trials.

- **`[1.1]` (Q1)** — the left panel is the test: observed AUC against the within-subject shuffled
  null. Inside the grey band = do not reject. The right panel is the honest strength check — a modest
  AUC that most subjects reproduce is stronger evidence than a large one carried by a few subjects.
  Judge by AUC and the direction count, **not** the p: with ~856 units the p is tiny for even a
  trivial difference, and two real cell classes always overlap heavily in rate.
- **`[2.1]` (Q2)** — `r > 0` means selection preferentially kept high-count cells within that class.
  Expected, and harmless on its own; only Q3 matters for the design.
- **`[2.2]` (Q3)** — the decision. `Δ` **inside** the null band = selection treated both classes
  alike, so the two selected subsets are comparable and a direct strength comparison on the category
  cells is fair. **Outside** = one class was filtered harder, and the truncation bias then runs
  *toward* the lower-rate class — the opposite direction to the prevalence bias.
- **`[3.1]`** — prints the branch and writes `costas_rate_diagnostic_<region>.csv` + `.json`.

Prevalence is **not** re-tested here; `[0.2]` prints the selected mix next to the base rate so they
can be read off the same line (they matched: 193 exc / 78 inh vs 184 / 87 expected).

**QC knob:** `MIN_WIN_SPIKES` (default 0 = keep everything). A cell firing a handful of spikes across
all presentations can never test as selective, so it pads the denominator without being informative;
`[0.2]` prints how many units fall under 50 in-window spikes if you want to set it.